# DSC0152 - notebook v66

## Kenapa v66 ada

v65 mendapat public **0.559**, turun dari **0.56312** milik v61. Membandingkan kedua run, selisihnya bukan hyper-parameter, tapi arsitektur.

Tiga hal yang saya ubah di v65 dan ketiganya menjauh dari yang menghasilkan skor public terbaik:

1. **v61 tidak memakai feature selection sama sekali.** Setiap kaki dilatih pada himpunan kolom yang didefinisikan tangan (`full`, `lean`, `fac`). v62 ke atas memakai seleksi gain per fold, dan semua run sejak itu memakainya.
2. **v66 membangun ulang blend v61 persis.** Rantai `trees4` → `core` → `tm30` → ekstrapolasi `tm30` menjauhi `lgb_lean` adalah satu-satunya mekanisme dengan dua hasil public terbukti: `qc_00` 0.56171 dan v61 0.56312. v65 tidak pernah membangun ulang `tm30`, dia mengekstrapolasi kaki terpilih menjauhi kaki tidak terpilih, dan itu objek yang berbeda.
3. **v61 merata-ratakan tiga LightGBM seed per kaki.** v65 memakai satu.

## Yang benar-benar diubah

Hanya jumlah training. Enam fold seedmenggantikan tiga, dan `lgbx_full` serta `linear` dipulihkan karena keduanya bagian dari rantai pemenang.

Resepnya diverifikasi lebih dulu, sebelum ada training baru: `alt_ext0.5_tm30_lean` dibangun ulang dari prediksi v61 yang tersimpan dan memberi KS **0.5286237**, cocok sampai tujuh desimal dengan `summary_v61_final.json`. Jadi rumus di `make_submission_v66.py` sudah diketahui benar sebelum Training baru dimulai.

## Public score yang pernah tercatat

| File | KS OOF | Public |
|---|---|---|
| v57 `aggressive_145` | 0.526988 | 0.55700 |
| v58ext `ext1.5` | 0.527511 | 0.55900 |
| tim `qc_00` | tidak terukur | 0.56171 |
| **v61 `alt_ext0.5_tm30_lean`** | 0.528624 | **0.56312** |
| v63 `consensus5_best` | **0.529867** | 0.55700 |
| v65 kandidat terbaik | 0.528592 | **0.55900** |

Pola itu konsisten: KS OOF tertinggi tidak menghasilkan public tertinggi. Rekor KS OOF adalah v63, dan public-nya justru salah yang terburuk.

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd

RUN_TRAINING = False
FOLDS = 10
FOLD_SEEDS = '42,7,13,21,5,101'
LGB_SEEDS = '0,1'

# The notebook lives in agas/v66/notebook/, so the working directory is not
# the repository root and Path.cwd() is wrong here. Walk up from this file.
HERE = Path.cwd().resolve()
if not (HERE / 'data').is_dir():
    for cand in [HERE, *HERE.parents]:
        if (cand / 'data').is_dir() and (cand / 'agas').is_dir():
            HERE = cand
            break
ROOT = HERE
AGAS = ROOT / 'agas'
VER = AGAS / 'v66'
OUT = VER / 'output'
SHARED = AGAS / 'shared'
TRAINING = VER / 'training'
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SHARED))
os.chdir(ROOT)
print('repo root  ', ROOT)
print('version dir', VER)

## 1. Verifikasi resep v61 dulu

Sebelum training apa pun, rumusWinner v61 dibangun ulang dari prediksi tersimpan. Kalau angkanya tidak cocok tujuh desimal, seluruh run ini tidak ada artinya.

In [ ]:
from scipy.stats import rankdata, ks_2samp
from sklearn.metrics import roc_auc_score

r = lambda v: rankdata(v) / len(v)

z = np.load(AGAS / 'v61/output/preds_v61.npz', allow_pickle=True)
y61 = pd.read_csv(ROOT / 'data/train_applications.csv',
                 usecols=['application_id', 'target']).set_index(
    'application_id').loc[z['train_ids']].target.values.astype(int)
ro = {k[4:]: r(z[k]) for k in z.files if k.startswith('oof_')}

trees = (ro['lgb_full'] + ro['lgb_lean'] + ro['cat_full'] + ro['xgb_full']) / 4
core = (trees * 2 + ro['lgbx_full']) / 3
tm30 = 0.7 * core + 0.3 * ro['linear']
alt = r(tm30 + 0.5 * (tm30 - ro['lgb_lean']))

print(f"rebuilt alt_ext0.5_tm30_lean  KS={ks_2samp(alt[y61==1], alt[y61==0]).statistic:.7f}  "
      f"AUC={roc_auc_score(y61, alt):.7f}")
print('target from summary_v61_final.json  KS=0.5286237  AUC=0.8466992')

## 2. Cacat nyata yang ditemukan saat Smoke test

Kaki `linear` gagal di smoke test dengan `SplineTransformer` yang menolak NaN. Penyebabnya bukan kode v66: dua kolom `af_` yang dibawa v65, yaitu `af_loan_to_hist_max` dan `af_loan_to_hist_mean`, **sepenuhnya NaN**. `median()` dari kolom kosong menghasilkan NaN, lalu `fillna(NaN)` tidak mengisi apa pun.

v61 tidak punya kedua kolom itu, jadi tidak pernah kena. Di v66 keduanya dibuang dari kaki linear, sekaligus mengembalikan kaki itu ke himpunan kolom yang sama dengan v61.

In [ ]:
# The v65 blocks live in the extra cache, so both caches have to be merged
# before the check can see them. Reading only the base cache finds nothing.
f = pd.read_pickle(SHARED / 'cache/features_cache.pkl')
ex = pd.read_pickle(SHARED / 'cache/extra_cache.pkl')
add = [c for c in ex.columns if c != 'application_id' and c not in f.columns]
f = f.merge(ex[['application_id'] + add], on='application_id', how='left')
allnan = [c for c in f.columns if c != 'application_id' and f[c].isna().all()]
print('all-NaN columns after merging both caches:', allnan)

## 3. Run utama

10 fold, enam fold seed, dua LightGBM seed per kaki, tujuh kaki. Parameter, himpunan kolom, dan daftar noise disalin verbatim dari v61.

In [ ]:
cmd = (f'python3 {TRAINING / "train_submission_v66.py"} --folds {FOLDS} '
       f'--fold-seeds {FOLD_SEEDS} --lgb-seeds {LGB_SEEDS} --out-prefix v66')
print(cmd)
if RUN_TRAINING:
    os.system(cmd)
else:
    print('dilewati, memakai prediksi v66 yang sudah ada')

s = json.loads((OUT / 'summary_v66.json').read_text())
legs = pd.DataFrame([{'leg': k, 'auc': v['auc'], 'ks': v['ks']}
                     for k, v in s.items() if isinstance(v, dict) and 'auc' in v])
print('minutes', round(s['minutes'], 1), '| columns full/lean/fac:',
      s['n_full'], s['n_lean'], s['n_fac'])
legs.sort_values('auc', ascending=False)

### Perbandingan kaki v66 dengan v61

v61 memakai tiga fold seed dan tiga LightGBM seed, v66 memakai enam dan dua. Yang diharapkan berubah adalah hanya pengurangan varians dari partisi fold, bukan kualitas model per partisi.

In [ ]:
v61s = json.loads((AGAS / 'v61/output/summary_v61.json').read_text())['models']
cmp = legs.merge(pd.DataFrame([{'leg': k, 'auc_v61': v['auc'], 'ks_v61': v['ks']}
                               for k, v in v61s.items()]), on='leg', how='inner')
cmp['auc_delta'] = (cmp.auc - cmp.auc_v61).round(6)
cmp['ks_delta'] = (cmp.ks - cmp.ks_v61).round(6)
cmp

## 4. Kandidat

Rantai v61 dibangun ulang utuh: `trees4`, `core`, `tm20`, `tm30`, lalu ekstrapolasi terhadap `lgb_lean` dan `fac_full` pada beberapa rasio. Yang ditambahkan hanya **plateau**: rata rank atas tiga rasio tetangga, karena rasio yang terbaik di v61 ada di plateau lebar dan satu rasio adalah pilihan yang paling mungkin merupakan undian noise.

KS tetap bukan kunci pengurutan. KS adalah statistik maksimum dengan sebaran sampling sekitar 0.0036 pada ukuran OOF, jadi argmax dari beberapa ratus vektor mengukur noise. AUC yang jadi kunci, KS dilaporkan dengan sebarannya dikurangi, dan vektor yang meranking hampir identik dibuang.

In [ ]:
if RUN_TRAINING:
    os.system(f'python3 {VER / "make_submission_v66.py"}')

cs = json.loads((OUT / 'summary_v66_candidates.json').read_text())
print('pool', cs['pool_size'], '-> eligible', cs['eligible'], '-> kept', cs['kept'])
pd.DataFrame(cs['candidates'])

## 5. Validasi

Tiap kandidat diperiksa ulang: 30000 baris, `application_id` unik dan sama dengan `data/test_applications.csv`, probabilitas di rentang [0, 1] tanpa NaN. Jumlah file di disk harus sama persis dengan jumlah entri di ringkasan, supaya nomor slot tidak pernah menunjuk ke isi berbeda antar run.

In [ ]:
import glob

ref = set(pd.read_csv(ROOT / 'data/test_applications.csv', usecols=['application_id']).application_id)
files = sorted(glob.glob(str(OUT / 'submission_v66_*.csv')))
assert {f.split('/')[-1] for f in files} == {c['file'] for c in cs['candidates']}

rows = []
for path in files:
    d = pd.read_csv(path)
    ok = (len(d) == 30000 and d.application_id.is_unique
          and set(d.application_id) == ref
          and d.default_probability.between(0, 1).all()
          and not d.default_probability.isna().any())
    rows.append({'file': path.split('/')[-1], 'rows': len(d), 'ok': ok})
res = pd.DataFrame(rows)
print('total', len(res), '| invalid', int((~res.ok).sum()))
res

## 6. Apa yang diketahui, dan apa yang tidak

**Diketahui.** Resep v61 bisa direproduksi tepat, jadi perbandingan v66 dengan v61 bersih. Modelnya sama persis, hanya lebih banyak partisi fold yang dirata-ratakan.

**Tidak diketahui.** Apakah lebih banyak fold seed memperbaiki public score. Yang jelas dari data: OOF KS dan public score hampir tidak berkorelasi di proyek ini, dan KS tertinggi sepanjang masa justru menghasilkan public terburuk. Jadi kemajuan OOF di run ini **bukan** bukti kemajuan public.

**Penjelasan yang paling mungkin untuk turunnya v65 ke 0.559.** v65tidak membangun `tm30`, tidak memakai `lgbx`, tidak memakai `linear`, memakai feature selection, dan memakai satu LightGBM seed.run v66 mengembalikan kelima hal itu. Kalau public score v66 masih di bawah 0.56312, kesimpulannya bukan "lebih banyak fold seed tidak berguna", tapi "v61 adalah titik lokal yang sulit dikalahkan dengan perubahan arsitektur", dan arah berikutnya bukan variasi lebih banyak melainkan mencari informasi yang belum dipakai.

Rekomendasi: kalau hanya satu yang dikirim, kirim kandidat dengan AUC tertinggi, karena AUC adalah ukuran kualitas ranking yang tidak menderita bias maksimum seperti KS. Catat public score aktual setiap file, karena itu satu-satunya ukuran yang benar.

In [ ]:
r = os.system('python3 agas/v64/verify_all_scripts.py')
print('verify exit code', r)